# Visualização: sífilis congênita em Porto Alegre

Este notebook cria uma visualização simples sobre desigualdade racial na sífilis congênita em Porto Alegre.

A pergunta central do gráfico é:

> Entre os casos notificados de sífilis congênita em que a mãe não realizou pré-natal, qual é o perfil de escolaridade dessas mães por grupo racial?

A análise foca no SINAN/SIFCBR, pois a pergunta combina desfecho, acesso ao pré-natal e escolaridade dentro dos casos notificados.


## 1. Dependências

O Google Colab inclui `pandas`, `matplotlib`, `seaborn` e `numpy`. As dependências adicionais deste notebook são `pyreaddbc`, para converter arquivos `.dbc` do DATASUS para `.dbf`, e `dbfread`, para leitura tabular do `.dbf` convertido.


In [ ]:
import importlib.util
import subprocess
import sys

pacotes = []
if importlib.util.find_spec("pyreaddbc") is None:
    pacotes.append("pyreaddbc==2.0.4")
if importlib.util.find_spec("dbfread") is None:
    pacotes.append("dbfread==2.0.7")

if pacotes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pacotes])
    print("Pacotes instalados:", ", ".join(pacotes))
else:
    print("Dependências já instaladas.")


## 2. Bibliotecas e configurações

Bibliotecas usadas para manipulação, preparação e visualização dos dados.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CODIGO_PORTO_ALEGRE = "431490"


## 3. Caminhos dos dados

A estrutura esperada no repositório é:

- `data/raw/SIFCBR24.dbc`
- `data/raw/sinasc/DNRS2024.dbc`
- `data/raw/cnes/STRS24*.dbc`


In [ ]:
def encontrar_arquivo(*candidatos):
    for candidato in candidatos:
        caminho = Path(candidato)
        if caminho.exists():
            return caminho
    caminhos = ", ".join(str(c) for c in candidatos)
    raise FileNotFoundError(f"Nenhum arquivo encontrado. Caminhos testados: {caminhos}")

ARQUIVO_SINAN = encontrar_arquivo(
    "../data/raw/SIFCBR24.dbc",
    "data/raw/SIFCBR24.dbc",
    "SIFCBR24.dbc",
)

ARQUIVO_SINASC = encontrar_arquivo(
    "../data/raw/sinasc/DNRS2024.dbc",
    "data/raw/sinasc/DNRS2024.dbc",
    "DNRS2024.dbc",
)

print("SINAN:", ARQUIVO_SINAN)
print("SINASC:", ARQUIVO_SINASC)


## 4. Leitura dos arquivos `.dbc`

O pacote `pyreaddbc` converte os arquivos comprimidos `.dbc` do DATASUS para `.dbf`. Em seguida, `dbfread` lê o `.dbf` convertido e o resultado é transformado em um DataFrame pandas.


In [ ]:
import tempfile
from pathlib import Path

import pyreaddbc
from dbfread import DBF, FieldParser


class ParserDatasInvalidas(FieldParser):
    def parseD(self, field, data):
        try:
            return super().parseD(field, data)
        except ValueError:
            return None


def ler_dbc(caminho_dbc, encoding="iso-8859-1"):
    caminho_dbc = Path(caminho_dbc)
    with tempfile.TemporaryDirectory() as tmpdir:
        caminho_dbf = Path(tmpdir) / f"{caminho_dbc.stem}.dbf"
        pyreaddbc.dbc2dbf(str(caminho_dbc), str(caminho_dbf))
        tabela = DBF(
            str(caminho_dbf),
            encoding=encoding,
            parserclass=ParserDatasInvalidas,
            load=True,
        )
        return pd.DataFrame(iter(tabela))


sinan = ler_dbc(ARQUIVO_SINAN)
sinasc = ler_dbc(ARQUIVO_SINASC)

print("SINAN", sinan.shape)
print("SINASC", sinasc.shape)


## 5. Recorte: Porto Alegre

Nos microdados, o código de Porto Alegre aparece como `431490` ou com dígito adicional. Por isso o filtro usa `startswith("431490")`.


In [ ]:
sinan_poa = sinan[
    sinan["ID_MN_RESI"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

sinasc_poa = sinasc[
    sinasc["CODMUNRES"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

print("Casos SINAN Porto Alegre:", len(sinan_poa))
print("Nascidos vivos SINASC Porto Alegre:", len(sinasc_poa))


## 6. Preparação da pergunta visual

Em vez de comparar indicadores soltos, esta etapa foca em uma pergunta única: nos casos de sífilis congênita em que não houve pré-natal, a baixa escolaridade aparece de forma desigual entre grupos raciais?

Para facilitar a leitura e evitar grupos muito pequenos, o recorte usa dois grupos:

- `Mães não negras`: brancas, amarelas e indígenas.
- `Mães negras`: pretas e pardas.

A escolaridade é agrupada em três níveis a partir do campo `ESCOLMAE` do SINAN/SIFCBR:

- até 7 anos de estudo;
- 8 anos ou mais de estudo;
- ignorada/sem informação.


In [ ]:
def grupo_raca_comparativo(valor):
    codigo = str(valor).strip()
    if codigo in {"2", "4"}:
        return "Mães negras"
    if codigo in {"1", "3", "5"}:
        return "Mães não negras"
    return "Ignorado/sem informação"


def grupo_escolaridade_sinan(valor):
    codigo = str(valor).strip()
    if codigo in {"02", "03", "04", "05"}:
        return "Até 7 anos de estudo"
    if codigo in {"06", "07", "08"}:
        return "8 anos ou mais de estudo"
    return "Ignorada/sem informação"


ordem_grupos = ["Mães não negras", "Mães negras"]
ordem_escolaridade = ["Até 7 anos de estudo", "8 anos ou mais de estudo", "Ignorada/sem informação"]

casos = sinan_poa.copy()
casos["grupo_raca"] = casos["ANT_RACA"].map(grupo_raca_comparativo)
casos["grupo_escolaridade"] = casos["ESCOLMAE"].map(grupo_escolaridade_sinan)
casos["pre_natal"] = casos["ANT_PRE_NA"].astype(str).str.strip()

casos_validos = casos[casos["grupo_raca"].isin(ordem_grupos)].copy()
casos_sem_pre_natal = casos_validos[casos_validos["pre_natal"].eq("2")].copy()

resumo_sem_pre_natal = (
    casos_sem_pre_natal
    .groupby(["grupo_raca", "grupo_escolaridade"], as_index=False)
    .size()
    .rename(columns={"size": "casos"})
)

base_completa = pd.MultiIndex.from_product(
    [ordem_grupos, ordem_escolaridade],
    names=["grupo_raca", "grupo_escolaridade"],
).to_frame(index=False)

resumo_sem_pre_natal = base_completa.merge(
    resumo_sem_pre_natal,
    on=["grupo_raca", "grupo_escolaridade"],
    how="left",
).fillna({"casos": 0})

resumo_sem_pre_natal["casos"] = resumo_sem_pre_natal["casos"].astype(int)
resumo_sem_pre_natal["total_grupo"] = resumo_sem_pre_natal.groupby("grupo_raca")["casos"].transform("sum")
resumo_sem_pre_natal["percentual"] = resumo_sem_pre_natal.apply(
    lambda row: 0 if row.total_grupo == 0 else row.casos / row.total_grupo * 100,
    axis=1,
)

resumo_sem_pre_natal["grupo_raca"] = pd.Categorical(
    resumo_sem_pre_natal["grupo_raca"], categories=ordem_grupos, ordered=True
)
resumo_sem_pre_natal["grupo_escolaridade"] = pd.Categorical(
    resumo_sem_pre_natal["grupo_escolaridade"], categories=ordem_escolaridade, ordered=True
)
resumo_sem_pre_natal = resumo_sem_pre_natal.sort_values(["grupo_raca", "grupo_escolaridade"])

print("Casos de sífilis congênita sem pré-natal por grupo racial:")
print(casos_sem_pre_natal["grupo_raca"].value_counts().reindex(ordem_grupos, fill_value=0))

resumo_sem_pre_natal


## 7. Tabela auxiliar para interpretação

A tabela mostra, entre os casos sem pré-natal, quantos casos existem em cada faixa de escolaridade e qual o percentual dentro do grupo racial.


In [ ]:
tabela_escolaridade = resumo_sem_pre_natal.copy()
tabela_escolaridade["percentual"] = tabela_escolaridade["percentual"].round(1)
tabela_escolaridade[["grupo_raca", "grupo_escolaridade", "casos", "total_grupo", "percentual"]]

## 8. Visualização final

O gráfico responde diretamente: entre os casos sem pré-natal, como a escolaridade se distribui em mães negras e mães não negras?

As barras são empilhadas em 100%, permitindo comparar a composição interna de cada grupo racial. Os rótulos mostram percentual e quantidade de casos.


In [ ]:
plot_data = resumo_sem_pre_natal.copy()
plot_data["rotulo"] = plot_data.apply(
    lambda row: "" if row.casos == 0 else f"{row.percentual:.1f}%\n(n={int(row.casos)})",
    axis=1,
)

fig, ax = plt.subplots(figsize=(11, 6.2))

cores = {
    "Até 7 anos de estudo": "#D55E00",
    "8 anos ou mais de estudo": "#0072B2",
    "Ignorada/sem informação": "#8A8F98",
}

acumulado = {grupo: 0 for grupo in ordem_grupos}

for escolaridade in ordem_escolaridade:
    subset = plot_data[plot_data["grupo_escolaridade"].eq(escolaridade)]
    valores = [float(subset[subset["grupo_raca"].eq(grupo)]["percentual"].iloc[0]) for grupo in ordem_grupos]
    bases = [acumulado[grupo] for grupo in ordem_grupos]

    barras = ax.barh(
        ordem_grupos,
        valores,
        left=bases,
        color=cores[escolaridade],
        label=escolaridade,
        edgecolor="white",
        linewidth=1,
    )

    for grupo, valor, base, barra in zip(ordem_grupos, valores, bases, barras):
        linha = subset[subset["grupo_raca"].eq(grupo)].iloc[0]
        if linha.casos > 0 and valor >= 8:
            ax.text(
                base + valor / 2,
                barra.get_y() + barra.get_height() / 2,
                f"{valor:.1f}%\n(n={int(linha.casos)})",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
                weight="bold",
            )
        elif linha.casos > 0:
            ax.text(
                base + valor + 1,
                barra.get_y() + barra.get_height() / 2,
                f"{valor:.1f}% (n={int(linha.casos)})",
                ha="left",
                va="center",
                fontsize=8,
                color="#333333",
            )
        acumulado[grupo] += valor

for i, grupo in enumerate(ordem_grupos):
    total = int(plot_data[plot_data["grupo_raca"].eq(grupo)]["total_grupo"].max())
    ax.text(101, i, f"Total: {total}", va="center", fontsize=9, color="#333333")

ax.set_title(
    "Casos de sífilis congênita sem pré-natal: escolaridade por grupo racial",
    fontsize=14,
    weight="bold",
    pad=14,
)
ax.set_xlabel("Distribuição percentual dentro dos casos sem pré-natal")
ax.set_ylabel("")
ax.set_xlim(0, 112)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.30), ncol=2, frameon=True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)

plt.tight_layout()

saida_dir = Path("../outputs/figures")
if not saida_dir.exists():
    saida_dir = Path("outputs/figures")
saida_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(saida_dir / "visualizacao_sifilis_congenita_poars.png", dpi=200, bbox_inches="tight")
plt.show()
